# Семинар №8. Программный доступ к БД из Python

## Как устроен ноутбук

1. Каждая тема идёт по схеме: **теория → демо → упражнение**.
2. Ячейки с комментарием `# УПРАЖНЕНИЕ N` — для самостоятельной работы; там стоят `TODO`.
3. Полные решения собраны в разделе «Решения» в самом конце. Старайтесь сначала решить сами.
4. Выполняйте ячейки **по порядку**: состояние таблиц в БД важно (мы переиспользуем одни и те же таблицы).
5. Сброс: `Kernel → Restart & Run All`.

## Окружение

- PostgreSQL 16 слушает `localhost:5432` **внутри контейнера**.
- Учётные данные БД: пользователь `postgres`, **пароль не требуется** (trust auth для локальных подключений), база `demo`.
- Jupyter-пароль: `student`.

## План

1. `psycopg` (DB-API): соединения, курсоры, параметризация.
2. Транзакции и `with conn:`.
3. SQL-инъекции и почему f-string в SQL — зло.
4. Батчевая вставка (`executemany`).
5. SQLAlchemy Core (`text()`).
6. Pandas: `read_sql` и `chunksize`.
7. ORM: модели, M:N, каскады.
8. Проблема N+1 и `selectinload`.


In [1]:
# Проверка окружения и общая конфигурация
import sys
print("Python:", sys.version.split()[0])

import psycopg, sqlalchemy, pandas, numpy
print("psycopg     :", psycopg.__version__)
print("sqlalchemy  :", sqlalchemy.__version__)
print("pandas      :", pandas.__version__)
print("numpy       :", numpy.__version__)

# Параметры подключения (внутри контейнера localhost = этот же контейнер).
# PostgreSQL настроен с trust-аутентификацией для локальных подключений,
# поэтому пароль НЕ требуется и НЕ передаётся.
DB = {
    "host":   "localhost",
    "port":   5432,
    "dbname": "demo",
    "user":   "postgres",
}

PSYCOPG_CONN = (
    f"host={DB['host']} port={DB['port']} dbname={DB['dbname']} "
    f"user={DB['user']}"
)
SQLALCHEMY_URL = (
    f"postgresql+psycopg://{DB['user']}@{DB['host']}:{DB['port']}/{DB['dbname']}"
)

print("\npsycopg DSN      :", PSYCOPG_CONN)
print("SQLAlchemy URL   :", SQLALCHEMY_URL)


Python: 3.12.16


psycopg     : 3.3.6
sqlalchemy  : 2.1.6
pandas      : 2.2.3
numpy       : 2.1.3

psycopg DSN      : host=localhost port=5432 dbname=demo user=postgres
SQLAlchemy URL   : postgresql+psycopg://postgres@localhost:5432/demo


## 1. Основы `psycopg` (DB-API)

**DB-API** (PEP 249) — стандарт Python для работы с реляционными БД: `connect()`, `cursor()`, `execute()`, `fetch*()`. У каждого драйвера свой диалект, но интерфейс общий.

Пакет `psycopg` (v3) — современный драйвер PostgreSQL. Что важно знать:

- **Соединение** открывается через `psycopg.connect(conn_string)`; его **обязательно** закрывать. Лучший способ — менеджер контекста `with psycopg.connect(...) as conn`.
- **Курсор** — объект, через который выполняются запросы; тоже умеет `with`.
- `cur.execute(sql, params)` — два аргумента: SQL-шаблон и кортеж/список параметров. Драйвер сам корректно экранирует и передаёт значения по бинарному протоколу. **Никогда не склеивайте SQL строками**.
- `cur.fetchone()` / `cur.fetchall()` / итерация по `cur` — способы получить результат.
- `psycopg.connect(..., row_factory=dict_row)` — строки как словари вместо кортежей.
- `psycopg.connect(...)` при выходе из `with` вызывает `close()`. Обратите внимание: `commit/rollback` на этом уровне **не** делается — это уже часть транзакций (см. тему 2).
- В реальных веб-приложениях напрямую `connect` почти не используют — берут пул (`psycopg_pool.ConnectionPool` или `QueuePool` в SQLAlchemy), потому что открытие нового TCP-соединения и форк бэкенд-процесса PG — дорого.


In [2]:
import psycopg
from psycopg.rows import dict_row

# --- 1. Версия сервера и типичный SELECT ---
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version()")
        print("PG version:", cur.fetchone()[0].split(" on ")[0])

# --- 2. DDL + DML + чтение в одной транзакции ---
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS demo_items")
        cur.execute("CREATE TABLE demo_items (id INT PRIMARY KEY, name TEXT, qty INT)")
        cur.execute(
            "INSERT INTO demo_items (id, name, qty) VALUES (%s, %s, %s), (%s, %s, %s)",
            (1, "apple", 10, 2, "banana", 20),
        )
    conn.commit()

    with conn.cursor() as cur:
        cur.execute("SELECT id, name, qty FROM demo_items ORDER BY id")
        print("\nrows (tuples):")
        for row in cur.fetchall():
            print(" ", row)

# --- 3. То же самое, но со словарями ---
with psycopg.connect(PSYCOPG_CONN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT id, name, qty FROM demo_items ORDER BY id")
        print("\nrows (dict_row):")
        for row in cur.fetchall():
            print(" ", row)


PG version: PostgreSQL 16.2

rows (tuples):
  (1, 'apple', 10)
  (2, 'banana', 20)

rows (dict_row):
  {'id': 1, 'name': 'apple', 'qty': 10}
  {'id': 2, 'name': 'banana', 'qty': 20}


### Упражнение 1. Параметризованный поиск

**Задание.** В таблице `demo_items` найдите запись, где `name` совпадает с `target_name`, и выведите `id, name, qty`.

**Шаги.**
1. Откройте соединение `psycopg.connect(PSYCOPG_CONN)` и курсор.
2. Выполните `SELECT id, name, qty FROM demo_items WHERE name = %s` с параметром `(target_name,)`.
3. Напечатайте результат.
4. Дополнительно: поменяйте `target_name` на `"apple' OR '1'='1"` и убедитесь, что запрос вернул 0 строк — инъекция не сработала.

**Критерий успеха.** При `target_name = 'apple'` печатается `(1, 'apple', 10)`; при «инъекционном» вводе — пусто.


In [3]:
# УПРАЖНЕНИЕ 1. Параметризованный поиск
#
# Ключевая идея: SQL и данные идут к серверу РАЗДЕЛЬНО.
#   * шаблон "WHERE name = %s" уходит на сервер как подготовленный запрос (Parse);
#   * значение target_name едет отдельным параметром (Bind) по расширенному протоколу;
#   * сервер НИКОГДА не разбирает содержимое параметра как SQL-код, поэтому даже
#     строка "apple' OR '1'='1" — это просто текст для сравнения, а не фрагмент запроса.

import psycopg  # драйвер PostgreSQL (DB-API, PEP 249)


def find_item(target_name):
    """Ищет товар по имени параметризованным запросом, возвращает список кортежей."""
    # Поведение psycopg.connect() по умолчанию (без доп. параметров):
    #   * autocommit=False -> транзакция открывается автоматически с первого execute(),
    #     уровень изоляции — READ COMMITTED (дефолт PostgreSQL);
    #   * row_factory не задан -> строки возвращаются КОРТЕЖАМИ
    #     (если передать row_factory=dict_row, как в демо выше, придут словари);
    #   * при выходе из `with psycopg.connect(...)` psycopg3 делает COMMIT
    #     (или ROLLBACK при исключении) и ЗАКРЫВАЕТ соединение.
    # Для SELECT коммитить нечего (транзакция read-only), закрытие корректно её завершает.
    with psycopg.connect(PSYCOPG_CONN) as conn:
        # Курсор — тоже менеджер контекста: закрывается при выходе из with.
        with conn.cursor() as cur:
            # Параметры передаются КОРТЕЖЕМ, даже если параметр один:
            # запятая в (target_name,) обязательна — иначе это не кортеж, а просто строка.
            cur.execute(
                "SELECT id, name, qty FROM demo_items WHERE name = %s",
                (target_name,),
            )
            return cur.fetchall()


# 1) Обычный ввод: ожидаем ровно одну строку (1, 'apple', 10)
print("поиск apple            :", find_item("apple"))

# 2) «Инъекционный» ввод: если бы значение вклеивалось в SQL (f-string),
#    условие превратилось бы в WHERE name = 'apple' OR '1'='1' и вернулось ВСЁ.
#    С параметром вернётся [] — инъекция не сработала.
print("поиск apple' OR '1'='1 :", find_item("apple' OR '1'='1"))

# ВЫВОД: критерий успеха выполнен — 'apple' находит (1, 'apple', 10),
# инъекционная строка не возвращает ничего: параметр не может стать кодом.


поиск apple            : [(1, 'apple', 10)]
поиск apple' OR '1'='1 : []


## 2. Транзакции и `with conn:`

- В PostgreSQL транзакция начинается с первого SQL-запроса.
- В psycopg3 у соединения есть метод `conn.transaction()`, но проще — **менеджер контекста `with conn:`**. Он делает `commit()` при успешном выходе и `rollback()` при любом исключении.
- **DDL транзакционен** (в отличие, например, от MySQL). Если вы упадёте после `CREATE TABLE` — таблица откатится.
- **Долгие транзакции — зло.** Пока транзакция открыта, PG не может почистить старые версии строк через `autovacuum` (MVCC, семинар 5). Отсюда bloat. Правило: транзакция должна длиться миллисекунды, а не минуты. Не держите транзакцию, пока ждёте ответа от медленного API.


In [4]:
from psycopg.errors import UniqueViolation

# Готовим таблицу ВНЕ проблемной транзакции
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS tx_demo")
        cur.execute("CREATE TABLE tx_demo (id INT PRIMARY KEY, name TEXT)")
    conn.commit()

try:
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            # Дубликат PK -> вторая строка упадёт
            cur.executemany("INSERT INTO tx_demo VALUES (%s, %s)", [(1, "a"), (1, "b")])
        conn.commit()
except UniqueViolation as e:
    print("Поймали UniqueViolation:", str(e).splitlines()[0])
    # with conn: автоматически сделает rollback

# Проверяем: в таблице 0 строк (вставка откатилась целиком)
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT count(*) FROM tx_demo")
        print("rows after rollback:", cur.fetchone()[0])


Поймали UniqueViolation: duplicate key value violates unique constraint "tx_demo_pkey"
rows after rollback: 0


### Упражнение 2. Атомарность при ошибке

**Задание.** Смоделируйте ситуацию, где часть строк вставляется успешно, а затем происходит ошибка, и убедитесь, что откатилось всё.

**Шаги.**
1. В отдельной транзакции создайте таблицу `ex2 (id INT PRIMARY KEY, name TEXT)` — без дубликатов.
2. В следующей транзакции выполните `executemany` со списком `[(1,'a'), (2,'b'), (1,'dup')]`.
3. Оберните в `try/except UniqueViolation`.
4. Проверьте `SELECT count(*) FROM ex2` — должно быть 0 (откатились даже успешные вставки).

**Критерий успеха.** Счётчик 0, в консоли — сообщение об ошибке.


In [5]:
# УПРАЖНЕНИЕ 2. Атомарность транзакции при ошибке
#
# Сценарий: часть строк вставляется успешно, затем ошибка -> откатиться должно ВСЁ.
# Это и есть атомарность: у транзакции в PostgreSQL нет «частичного commit».
#
# Почему транзакции устроены в этом коде именно так:
#   * DDL (DROP/CREATE) вынесен в ОТДЕЛЬНУЮ транзакцию. DDL в PostgreSQL транзакционен
#     (в отличие от MySQL), но смешивать его с данными — плохая практика:
#     DDL берёт более тяжёлые блокировки, а откат данных потянул бы за собой и схему.
#   * Уровень изоляции — READ COMMITTED (дефолт PG, явно не указываем): у нас одна
#     сессия и нет конкурентных писателей, нам важна атомарность, а не согласованный
#     снапшот — READ COMMITTED здесь достаточен.
#   * Каждая транзакция открывается автоматически с первого execute()
#     (autocommit=False по умолчанию) и закрывается: либо явным conn.commit(),
#     либо при выходе из `with psycopg.connect(...)` -> commit/rollback + close().

from psycopg.errors import UniqueViolation  # класс ошибки PG 23505 (нарушение PK/UNIQUE)

# --- Шаг 1. Таблица — отдельной транзакцией, без дубликатов -----------
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS ex2")   # идемпотентность: ячейку можно перезапускать
        cur.execute("CREATE TABLE ex2 (id INT PRIMARY KEY, name TEXT)")
    conn.commit()   # <-- здесь ЗАКРЫВАЕТСЯ транзакция с DDL
    # при выходе из with соединение закроется (close())

# --- Шаги 2-3. Транзакция с ошибкой посередине ------------------------
try:
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            # executemany выполнит 3 INSERT в ОДНОЙ транзакции:
            # (1,'a') и (2,'b') вставятся, (1,'dup') нарушит PRIMARY KEY.
            cur.executemany(
                "INSERT INTO ex2 VALUES (%s, %s)",
                [(1, "a"), (2, "b"), (1, "dup")],
            )
        conn.commit()   # сюда не дойдём: исключение вылетит на executemany
    # если бы дошли: COMMIT + close() при выходе из with
except UniqueViolation as e:
    # Исключение вылетело из `with psycopg.connect(...)` -> менеджер контекста
    # сделал ROLLBACK (откатились и «успешные» строки тоже) и ЗАКРЫЛ соединение.
    print("Поймали UniqueViolation:", str(e).splitlines()[0])
    # Если бы нужно было СОХРАНИТЬ часть строк, а откатить только «хвост»,
    # пригодились бы savepoints: `with conn.transaction(): ...` внутри соединения.

# --- Шаг 4. Проверка в новой транзакции -------------------------------
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT count(*) FROM ex2")
        print("Строк в ex2 после отката:", cur.fetchone()[0])   # 0 — откатилось ВСЁ


Поймали UniqueViolation: duplicate key value violates unique constraint "ex2_pkey"
Строк в ex2 после отката: 0


## 3. SQL-инъекции и параметризация

**SQL-инъекция** — это когда пользовательский ввод попадает в строку SQL и интерпретируется как код.

- ❌ **Уязвимо:** `cur.execute(f"... WHERE name = '{name}'")`. Если `name = "' OR '1'='1"`, запрос вернёт все строки; если `name = "'; DROP TABLE ...; --"` — таблица исчезнет.
- ✅ **Безопасно:** `cur.execute("... WHERE name = %s", (name,))`. Данные и SQL идут раздельно, значения передаются по бинарному протоколу и **не могут быть интерпретированы как код**.
- То же в SQLAlchemy Core: `text("... WHERE name = :name")` + `{"name": ...}`. **Никогда** не подставляйте значения внутрь `text()` через f-строку.


In [6]:
def unsafe_search(name):
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            # ❌ f-string — SQL-инъекция
            cur.execute(f"SELECT id, name, qty FROM demo_items WHERE name = '{name}'")
            return cur.fetchall()

def safe_search(name):
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            # ✅ параметризация
            cur.execute("SELECT id, name, qty FROM demo_items WHERE name = %s", (name,))
            return cur.fetchall()

malicious = "' OR '1'='1' --"
print("unsafe :", unsafe_search(malicious))
print("safe   :", safe_search(malicious))


unsafe : [(1, 'apple', 10), (2, 'banana', 20)]
safe   : []


### Упражнение 3. Почините уязвимый запрос

**Задание.** Дана функция `broken(code)` — она ищет пилота по коду через f-string. Перепишите её безопасно.

**Шаги.**
1. Скопируйте функцию, замените f-string на `%s` + параметр.
2. Проверьте: при `code = "' OR '1'='1"` результат должен быть пустым.
3. Дополнительно: добавьте `LIMIT 5` в SQL.

**Критерий успеха.** `broken("' OR '1'='1")` (старая версия) вернёт всё, `fixed("' OR '1'='1")` — пусто.


In [7]:
# УПРАЖНЕНИЕ 3. Чиним уязвимый запрос
#
# Почему broken() опасна: f-string вклеивает пользовательский ввод В ТЕЛО SQL.
# Ввод "' OR '1'='1" закрывает кавычку литерала и добавляет всегда-истинное условие:
#     WHERE name = '' OR '1'='1'   ->   вернутся ВСЕ строки.
# А ввод "'; DROP TABLE demo_items; --" (в подходящем окружении) снёс бы таблицу.

def broken(code):
    # ❌ уязвимая версия: данные смешаны с кодом запроса
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT id, name FROM demo_items WHERE name = '{code}'")
            return cur.fetchall()

def fixed(code):
    # ✅ безопасная версия: SQL-шаблон и параметр идут раздельно (расширенный протокол),
    # поэтому значение code НИКОГДА не интерпретируется как SQL.
    # LIMIT 5 — «защита в глубину»: даже если совпадений много, наружу уйдёт не больше 5 строк.
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            cur.execute(
                "SELECT id, name, qty FROM demo_items WHERE name = %s LIMIT 5",
                (code,),   # параметр — кортежем
            )
            return cur.fetchall()

malicious = "' OR '1'='1"
print("broken (уязвимо)   :", broken(malicious))   # вернул ВСЮ таблицу — инъекция сработала
print("fixed  (безопасно) :", fixed(malicious))    # [] — параметр не стал кодом

# Правило на всю жизнь: пользовательский ввод попадает в SQL ТОЛЬКО через плейсхолдеры
# (%s в psycopg, :name в SQLAlchemy). Никаких f-string, конкатенации и .format().


broken (уязвимо)   : [(1, 'apple'), (2, 'banana')]
fixed  (безопасно) : []


## 4. Батчевая вставка: `executemany`

- Цикл `for row in rows: cur.execute("INSERT ...", row)` — это **N сетевых round-trip'ов** и N записей в WAL. Медленно.
- `cur.executemany(sql, list_of_tuples)` — один вызов, драйвер группирует строки в пакеты.
- В `psycopg3` (≥3.1) `executemany` дополнительно объединяет несколько `INSERT` в один `pipeline`-пакет.
- Для по-настоящему больших объёмов (миллионы строк) есть отдельный, ещё более быстрый протокол — `COPY` (`cur.copy()`), но `executemany` — уже огромный шаг вперёд по сравнению с циклом.


In [8]:
import time, random

N = 3000
rows = [(i, f"item_{i}", random.randint(1, 100)) for i in range(1, N + 1)]

# Готовим таблицу
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS bench")
        cur.execute("CREATE TABLE bench (id INT PRIMARY KEY, name TEXT, val INT)")
    conn.commit()

# 1) Цикл
t0 = time.perf_counter()
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        for r in rows:
            cur.execute("INSERT INTO bench VALUES (%s, %s, %s)", r)
    conn.commit()
t_loop = time.perf_counter() - t0
print(f"цикл         : {t_loop:.2f}s")

# 2) executemany
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("TRUNCATE bench")
    conn.commit()

t0 = time.perf_counter()
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.executemany("INSERT INTO bench VALUES (%s, %s, %s)", rows)
    conn.commit()
t_batch = time.perf_counter() - t0
print(f"executemany  : {t_batch:.2f}s")
print(f"ускорение    : x{t_loop / max(t_batch, 1e-9):.1f}")


цикл         : 0.14s


executemany  : 0.03s
ускорение    : x4.6


### Упражнение 4. Батч на 10 000 строк

**Задание.** Сравните цикл и `executemany` на 10 000 строк и найдите порог, на котором `executemany` стабильно быстрее.

**Шаги.**
1. Сгенерируйте 10 000 кортежей `(id, name, val)`.
2. Замерьте цикл, замерьте `executemany`.
3. Напечатайте `speedup = t_loop / t_batch`.
4. **Вопрос на подумать:** почему `executemany` не «в N раз быстрее» линейно?

**Критерий успеха.** `speedup > 5`. В комментарии — 1–2 строки объяснения.


In [9]:
# УПРАЖНЕНИЕ 4. Цикл vs executemany: 10 000 строк и поиск порога
import time
import random

random.seed(42)   # фиксируем зерно: замеры воспроизводимы от запуска к запуску

TABLE = "bench10k"   # f-string ниже безопасна: имя таблицы — НАША константа, а не
                     # пользовательский ввод; значения строк всегда идут через %s

# Схему создаём отдельной транзакцией (DDL): commit здесь, close() при выходе из with
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute(f"DROP TABLE IF EXISTS {TABLE}")
        cur.execute(f"CREATE TABLE {TABLE} (id INT PRIMARY KEY, name TEXT, val INT)")
    conn.commit()

rows_10k = [(i, f"item_{i}", random.randint(1, 100)) for i in range(1, 10_001)]

def clean(dsn=PSYCOPG_CONN):
    """TRUNCATE выносим из замера: сбрасываем таблицу отдельной транзакцией."""
    with psycopg.connect(dsn) as conn:
        with conn.cursor() as cur:
            cur.execute(f"TRUNCATE {TABLE}")
        conn.commit()

def run_loop(rows, dsn=PSYCOPG_CONN):
    """N отдельных cur.execute() = N round-trip'ов + N раз разбор/планирование запроса."""
    clean(dsn)
    t0 = time.perf_counter()
    with psycopg.connect(dsn) as conn:
        with conn.cursor() as cur:
            for r in rows:
                cur.execute("INSERT INTO bench10k VALUES (%s, %s, %s)", r)
        conn.commit()   # один commit на все вставки; close() при выходе из with
    return time.perf_counter() - t0

def run_executemany(rows, dsn=PSYCOPG_CONN):
    """Один cur.executemany(): psycopg3 (>=3.1) сам собирает INSERT в pipeline-пакеты."""
    clean(dsn)
    t0 = time.perf_counter()
    with psycopg.connect(dsn) as conn:
        with conn.cursor() as cur:
            cur.executemany("INSERT INTO bench10k VALUES (%s, %s, %s)", rows)
        conn.commit()
    return time.perf_counter() - t0

def best_of(fn, rows, times=2):
    """Лучший из двух прогонов — гасим случайные паузы (GC, autovacuum, диск)."""
    return min(fn(rows) for _ in range(times))

print(f"{'N':>6} | {'цикл, c':>8} | {'executemany, c':>14} | {'ускорение':>9}")
for n in (10, 100, 1_000, 10_000):
    rows = rows_10k[:n]
    t_loop = best_of(run_loop, rows)
    t_batch = best_of(run_executemany, rows)
    print(f"{n:>6} | {t_loop:>8.3f} | {t_batch:>14.3f} | x{t_loop / max(t_batch, 1e-9):>8.1f}")

# ВОПРОС НА ПОДУМАТЬ: почему executemany НЕ «в N раз» быстрее?
# 1) У обоих способов есть постоянная часть: открытие соединения, BEGIN/COMMIT, fsync WAL.
# 2) Сервер всё равно вставляет строки ПО ОДНОЙ (executemany — это не один
#    multi-VALUES INSERT), поэтому запись данных, WAL и индексов никуда не исчезает.
# 3) Экономим только клиент-серверную «накладку»: round-trip'ы и разбор каждого INSERT.
# Отсюда два следствия, оба видны в замерах выше:
#   * ПОРОГ: на 10 строках выигрыша почти нет, со ~100 строк executemany стабильно быстрее;
#   * ПОТОЛОК: на localhost round-trip стоит микросекунды, поэтому ускорение тут
#     ограничено (~x4-5): экономить почти нечего. Проверим это явно бонусом ниже.
#
# БОНУС. Тот же замер на 10 000 строк через «медленную сеть»: локальный TCP-прокси
# добавляет 0.2 мс на каждый пакет в каждую сторону. Если executemany экономит
# именно round-trip'ы, в медленной сети выигрыш должен вырасти на порядок.
import socket
import threading

def latency_proxy(listen_port, target_port, delay, stop_event):
    """Простейший TCP-прокси: перекладывает байты, спя delay на каждый чанк."""
    srv = socket.socket()
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind(("127.0.0.1", listen_port))
    srv.listen(16)
    srv.settimeout(0.2)
    while not stop_event.is_set():
        try:
            client, _ = srv.accept()
        except socket.timeout:
            continue
        upstream = socket.socket()
        upstream.connect(("127.0.0.1", target_port))

        def pump(src, dst):
            try:
                while True:
                    data = src.recv(65536)
                    if not data:
                        break
                    time.sleep(delay)   # <-- искусственная задержка «сети»
                    dst.sendall(data)
            except OSError:
                pass
            finally:
                try:
                    dst.shutdown(socket.SHUT_WR)
                except OSError:
                    pass

        threading.Thread(target=pump, args=(client, upstream), daemon=True).start()
        threading.Thread(target=pump, args=(upstream, client), daemon=True).start()
    srv.close()

DSN_LAT = "host=127.0.0.1 port=5433 dbname=demo user=postgres"   # тот же PG, но «далеко»
stop_event = threading.Event()
threading.Thread(target=latency_proxy, args=(5433, 5432, 0.0002, stop_event), daemon=True).start()
time.sleep(0.3)   # даём прокси подняться
try:
    t_loop = run_loop(rows_10k, DSN_LAT)           # ~10 000 round-trip'ов, каждый подорожал
    t_batch = run_executemany(rows_10k, DSN_LAT)   # pipeline: от задержки почти не зависит
    print()
    print("10 000 строк через «медленную сеть» (0.2 мс на пакет):")
    print(f"цикл = {t_loop:.2f}s, executemany = {t_batch:.2f}s, ускорение = x{t_loop / max(t_batch, 1e-9):.0f}")
finally:
    stop_event.set()

# ВЫВОД: executemany экономит round-trip'ы, а не работу сервера. Поэтому выигрыш
# растёт с ценой сети: на localhost — x4-5, в сети с задержкой — x20 и выше.


     N |  цикл, c | executemany, c | ускорение
    10 |    0.003 |          0.002 | x     1.1
   100 |    0.006 |          0.003 | x     2.1


  1000 |    0.038 |          0.011 | x     3.4


 10000 |    0.479 |          0.116 | x     4.1



10 000 строк через «медленную сеть» (0.2 мс на пакет):
цикл = 7.97s, executemany = 0.21s, ускорение = x38


## 5. SQLAlchemy Core

**Core** — это слой выражений SQL поверх DB-API. Даёт:

- **Пул соединений** из коробки (`QueuePool`).
- **Параметризацию:** `text("WHERE x = :x")` + `[{...}, {...}]` — автоматически превращается в `executemany`.
- Единый интерфейс к PostgreSQL / SQLite / MySQL.

Ключевые вещи:
- `create_engine(url, echo=False)` — движок, управляет пулом. `echo=True` печатает SQL в консоль (обязательно при отладке ORM).
- `text("SELECT ... :param")` — обёртка над строкой SQL. Параметры — именованные `:name`. **f-строки внутрь `text()` нельзя.**
- `engine.connect()` → `conn.execute(stmt, params)` → `conn.commit()`.
- Если второй аргумент `execute` — список словарей, Core сам выполнит `executemany`.


In [10]:
from sqlalchemy import create_engine, text

engine = create_engine(SQLALCHEMY_URL, echo=False)

# --- DDL + батчевая вставка через Core ---
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS core_demo"))
    conn.execute(text("CREATE TABLE core_demo (id INT PRIMARY KEY, sensor TEXT, val NUMERIC)"))
    conn.commit()

    records = [
        {"id": i, "s": f"SENSOR_{i % 7}", "v": round((i * 0.37) % 100, 2)}
        for i in range(1, 501)
    ]
    # список dict -> автоматически executemany
    conn.execute(
        text("INSERT INTO core_demo (id, sensor, val) VALUES (:id, :s, :v)"),
        records,
    )
    conn.commit()
    print(f"Вставлено {len(records)} строк через Core.executemany")

    # Простой агрегат через Core
    row = conn.execute(
        text("SELECT count(*), max(val) FROM core_demo WHERE val > :threshold"),
        {"threshold": 50},
    ).one()
    print(f"Строк с val > 50: {row[0]}, максимум: {row[1]}")


Вставлено 500 строк через Core.executemany
Строк с val > 50: 230, максимум: 99.9


### Упражнение 5. Core: создать таблицу и залить батч

**Задание.** Через `text()` и `engine.connect()` создайте таблицу `ex5_measurements (id, sensor, value)` и вставьте 200 строк списком словарей.

**Шаги.**
1. `DROP TABLE IF EXISTS ex5_measurements` и `CREATE TABLE ...`.
2. Сгенерируйте `records` — список словарей `{"id", "sensor", "value"}` на 200 элементов.
3. Один вызов `conn.execute(text("INSERT ..."), records)`.
4. Проверьте: `SELECT count(*) FROM ex5_measurements WHERE value > 50`.

**Критерий успеха.** В консоли: `count > 0`. В логах — один `INSERT` (executemany), не 200.


In [11]:
# УПРАЖНЕНИЕ 5. SQLAlchemy Core: таблица + батч из 200 строк
#
# engine уже создан в демо выше (create_engine(SQLALCHEMY_URL)) — переиспользуем ЕГО:
# движок и его пул соединений (QueuePool) должны существовать в единственном
# экземпляре на приложение. Поведение по умолчанию: pool_size=5 (+10 overflow), echo=False.
# engine.connect() не открывает новое TCP-соединение, а выдаёт свободное из пула.

from sqlalchemy import event, text

with engine.connect() as conn:
    # Транзакция №1: DDL. Начинается автоматически с первого conn.execute(),
    # закрываем ЯВНЫМ conn.commit() — чтобы DDL не смешивался с данными.
    conn.execute(text("DROP TABLE IF EXISTS ex5_measurements"))
    conn.execute(text("CREATE TABLE ex5_measurements (id INT PRIMARY KEY, sensor TEXT, value NUMERIC)"))
    conn.commit()

    records = [
        {"id": i, "sensor": f"S{i % 4}", "value": round(i * 0.7, 2)}
        for i in range(1, 201)
    ]

    # Транзакция №2: данные. Список словарей во втором аргументе execute()
    # = executemany под капотом: шаблон подготовлен один раз, в него подставлено
    # 200 наборов параметров :id, :sensor, :value. Значения идут СТРОГО через
    # параметры — SQL-инъекция невозможна по построению.
    # Проверим «INSERT один, а не 200» счётчиком вызовов курсора (эквивалент echo=True):
    calls = {"n": 0, "many": False}

    def _count(conn, cursor, statement, params, ctx, many):
        calls["n"] += 1
        calls["many"] = calls["many"] or many

    event.listen(engine, "before_cursor_execute", _count)
    try:
        conn.execute(
            text("INSERT INTO ex5_measurements (id, sensor, value) VALUES (:id, :sensor, :value)"),
            records,
        )
    finally:
        event.remove(engine, "before_cursor_execute", _count)
    conn.commit()   # <-- здесь закрывается транзакция с данными
    print(f"200 строк одним вызовом: SQL-вызовов = {calls['n']}, executemany = {calls['many']}")

    # Транзакция №3: read-only SELECT. Коммитить нечего: при выходе из with
    # соединение вернётся в пул, SQLAlchemy по умолчанию сделает reset (rollback).
    cnt = conn.execute(
        text("SELECT count(*) FROM ex5_measurements WHERE value > :t"),
        {"t": 50},
    ).scalar()
    print("Строк со value > 50:", cnt)

# При выходе из `with engine.connect()` соединение НЕ закрывается,
# а ВОЗВРАЩАЕТСЯ В ПУЛ для переиспользования — в этом и смысл пула.


200 строк одним вызовом: SQL-вызовов = 1, executemany = True
Строк со value > 50: 129


## 6. Pandas + `chunksize`

`pandas` — in-memory аналитика. **DataFrame** — это таблица в RAM.

- `pd.read_sql(query, engine, chunksize=500)` возвращает **итератор** по чанкам. Так вы не выгрузите 10 ГБ в 8 ГБ RAM.
- `np.where(cond, a, b)` — векторизованный `CASE WHEN`. Работает на C, а не в Python-цикле.
- `chunk[chunk['col'] > 0]` — фильтрация тоже векторизована.

**Разделение труда:** PostgreSQL — для OLTP и сложных `JOIN`. Pandas — для математических трансформаций, feature engineering, агрегатов «в памяти».


In [12]:
import pandas as pd
import numpy as np

total = {"HIGH": 0, "LOW": 0}
for chunk in pd.read_sql("SELECT * FROM core_demo", engine, chunksize=200):
    chunk["level"] = np.where(chunk["val"] > 50, "HIGH", "LOW")
    vc = chunk["level"].value_counts()
    for k in total:
        total[k] += int(vc.get(k, 0))
print("По всей таблице:", total)
print("Строк всего   :", sum(total.values()))


По всей таблице: {'HIGH': 230, 'LOW': 270}
Строк всего   : 500


### Упражнение 6. Агрегат по сенсорам

**Задание.** Читая `ex5_measurements` чанками по 50, посчитайте среднее `value` и количество замеров по каждому `sensor`.

**Шаги.**
1. Итерируйтесь `pd.read_sql(..., engine, chunksize=50)`.
2. Для каждого чанка считайте `groupby("sensor").agg(mean=("value","mean"), n=("value","size"))`.
3. Соберите результаты в словарь `{sensor: {"sum": ..., "n": ...}}` (агрегируйте среднее, пересчитывая взвешенно).
4. Напечатайте итоговую таблицу.

**Критерий успеха.** Сумма `n` по всем сенсорам == общее число строк в `ex5_measurements`.


In [13]:
# УПРАЖНЕНИЕ 6. Агрегат по сенсорам: read_sql с chunksize
import pandas as pd

# pd.read_sql(..., chunksize=50) возвращает ИТЕРАТОР по чанкам-DataFrame'ам:
# в памяти одновременно живёт только один чанк (50 строк), а не вся таблица.
# Поэтому тот же паттерн работает и на таблицах, которые в RAM не влезут.
agg = {}   # {sensor: {"sum": сумма value, "n": число замеров}}

# Запрос — константа. Если бы он собирался из пользовательского ввода,
# параметризовали бы через text("... WHERE sensor = :s") + {"s": ...}.
for chunk in pd.read_sql(
    "SELECT sensor, value FROM ex5_measurements",
    engine,
    chunksize=50,
):
    # psycopg отдаёт NUMERIC как Decimal; на этом пути pandas сам приводит колонку
    # к float64. Явный astype(float) — дешёвая страховка: не сломается, даже если
    # драйвер вернёт Decimal (object dtype), и остаётся векторизованным.
    chunk["value"] = chunk["value"].astype(float)

    # группировка ВНУТРИ чанка: среднее и число строк по каждому сенсору
    g = chunk.groupby("sensor").agg(mean=("value", "mean"), n=("value", "size"))

    for sensor, row in g.iterrows():
        if sensor not in agg:
            agg[sensor] = {"sum": 0.0, "n": 0}
        # Средние по чанкам усреднять «в лоб» нельзя (чанки разных размеров),
        # поэтому копим СУММУ и ЧИСЛО строк: итог = sum / n (взвешенное среднее).
        agg[sensor]["sum"] += float(row["mean"]) * int(row["n"])
        agg[sensor]["n"] += int(row["n"])

print("Итог по сенсорам:")
for sensor, a in sorted(agg.items()):
    print(f"  {sensor}: mean = {a['sum'] / a['n']:.2f}, n = {a['n']}")

# Критерий успеха: сумма n по всем сенсорам == общее число строк в таблице
total_n = sum(a["n"] for a in agg.values())
with engine.connect() as conn:
    db_n = conn.execute(text("SELECT count(*) FROM ex5_measurements")).scalar()
print(f"сумма n = {total_n}, строк в таблице = {db_n} -> совпадает: {total_n == db_n}")


Итог по сенсорам:
  S0: mean = 71.40, n = 50
  S1: mean = 69.30, n = 50
  S2: mean = 70.00, n = 50
  S3: mean = 70.70, n = 50
сумма n = 200, строк в таблице = 200 -> совпадает: True


## 7. SQLAlchemy ORM

ORM мапит таблицы на Python-классы. Удобно для CRUD, но требует дисциплины.

- Наследуемся от `DeclarativeBase`; колонки описываем через `Mapped[T] = mapped_column(...)`.
- **M:N** — через ассоциативную `Table`, в `relationship(secondary=assoc)`.
- **Каскад** — `ondelete='CASCADE'` **на уровне БД** (в `ForeignKey`). Это быстрее, чем ORM-каскад: PG удалит сам, без выборки всех дочерних строк в память.
- `with Session(engine) as s:` — сессия. `s.add(obj)`, `s.commit()`, `s.delete(obj)`, `s.scalars(select(...))`.
- `engine.echo = True` — печатать SQL в консоль (включайте при отладке!).


In [14]:
from sqlalchemy import ForeignKey, String, Table, Column, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session

class Base(DeclarativeBase):
    pass

pilot_aircraft = Table(
    "pilot_aircraft", Base.metadata,
    Column("pilot_id", ForeignKey("pilots.id", ondelete="CASCADE"), primary_key=True),
    Column("aircraft_code", ForeignKey("aircrafts_orm.aircraft_code", ondelete="CASCADE"), primary_key=True),
)

class Aircraft(Base):
    __tablename__ = "aircrafts_orm"
    aircraft_code: Mapped[str] = mapped_column(String(3), primary_key=True)
    model:         Mapped[str] = mapped_column(String(50))
    pilots: Mapped[list["Pilot"]] = relationship(secondary=pilot_aircraft, back_populates="aircrafts")

class Pilot(Base):
    __tablename__ = "pilots"
    id:   Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    aircrafts: Mapped[list["Aircraft"]] = relationship(secondary=pilot_aircraft, back_populates="pilots")

Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

with Session(engine) as s:
    boeing = Aircraft(aircraft_code="773", model="Boeing 777-300")
    p1 = Pilot(name="Иван", aircrafts=[boeing])
    s.add(p1)
    s.commit()

    # Каскадное удаление
    to_del = s.query(Pilot).filter_by(name="Иван").first()
    s.delete(to_del)
    s.commit()

    # Связей не осталось?
    links = s.execute(select(pilot_aircraft)).fetchall()
    print("Связей после каскада:", len(links))   # 0


Связей после каскада: 0


### Упражнение 7. Своя M:N-модель

**Задание.** Опишите M:N между `Author` и `Book` через ассоциативную `author_book`.

**Шаги.**
1. Объявите `Author` (id, name) и `Book` (id, title) и `author_book` через `Table`.
2. В `relationship` пропишите `secondary=author_book` и `back_populates` с обеих сторон.
3. `create_all`.
4. Добавьте 2 авторов и 3 книги (некоторые — с двумя авторами).
5. Удалите одного автора и проверьте, что связи в `author_book` для него исчезли, а книги, у которых были другие авторы, остались.

**Критерий успеха.** До удаления связей 3–4, после удаления — меньше на число связей удалённого автора.


In [15]:
# УПРАЖНЕНИЕ 7. M:N «авторы — книги» через ассоциативную таблицу + каскад
from sqlalchemy import ForeignKey, String, Table, Column, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session

# Отдельный Base со своей metadata — чтобы не пересекаться с таблицами семинара
# (pilots/aircrafts_orm): у каждого DeclarativeBase свой реестр таблиц.
class Base2(DeclarativeBase):
    pass

# Ассоциативная таблица для M:N: ровно два FK, вместе образуют составной PRIMARY KEY.
# ondelete="CASCADE" прописан на уровне БД: если автора удалят «мимо» ORM
# (raw SQL, bulk-DELETE), PostgreSQL сам почистит строки связи без участия Python.
author_book = Table(
    "author_book", Base2.metadata,
    Column("author_id", ForeignKey("authors.id", ondelete="CASCADE"), primary_key=True),
    Column("book_id",   ForeignKey("books.id",   ondelete="CASCADE"), primary_key=True),
)

class Author(Base2):
    __tablename__ = "authors"
    id:   Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    # secondary=... — связь через ассоциативную таблицу;
    # back_populates — двусторонняя синхронизация: добавил книгу автору ->
    # у книги появился этот автор (и наоборот).
    books: Mapped[list["Book"]] = relationship(secondary=author_book, back_populates="authors")

class Book(Base2):
    __tablename__ = "books"
    id:    Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(200))
    authors: Mapped[list["Author"]] = relationship(secondary=author_book, back_populates="books")

# Идемпотентная подготовка схемы: drop_all/create_all через тот же пул engine.
Base2.metadata.drop_all(engine)
Base2.metadata.create_all(engine)

with Session(engine) as s:
    # «Война и мир» — ОДИН общий объект у ДВУХ авторов: классический M:N.
    war_and_peace = Book(title="Война и мир")
    tolstoy = Author(name="Толстой",     books=[war_and_peace, Book(title="Анна Каренина")])
    dosto   = Author(name="Достоевский", books=[war_and_peace, Book(title="Идиот")])
    s.add_all([tolstoy, dosto])
    s.commit()   # ОДНА транзакция: INSERT authors, INSERT books, INSERT author_book

    n_links_before = len(s.execute(select(author_book)).fetchall())
    print("Связей до удаления   :", n_links_before)          # 4
    print("Книг до удаления     :", s.query(Book).count())   # 3

    # Удаляем Толстого. Нюанс, ради которого стоит заглянуть в echo=True:
    # при удалении через Session ORM САМ удаляет его строки из author_book
    # (DELETE FROM author_book WHERE author_id=...) до DELETE FROM authors.
    # ondelete="CASCADE" остаётся страховкой на уровне БД — она срабатывает при
    # удалениях вне ORM (raw SQL, bulk delete). Если хочется, чтобы связи всегда
    # чистила именно БД, у relationship ставят passive_deletes=True.
    s.delete(s.query(Author).filter_by(name="Толстой").first())
    s.commit()

    n_links_after = len(s.execute(select(author_book)).fetchall())
    print("Связей после удаления:", n_links_after)           # 2 (у Толстого было 2)
    print("Книг осталось        :", s.query(Book).count())  # 3 — книги не тронуты
    wam = s.query(Book).filter_by(title="Война и мир").first()
    print("Автор(ы) «Войны и мира»:", [a.name for a in wam.authors])  # только Достоевский


Связей до удаления   : 4


Книг до удаления     : 3
Связей после удаления: 2
Книг осталось        : 3
Автор(ы) «Войны и мира»: ['Достоевский']


## 8. Проблема N+1 и `selectinload`

**N+1** — это когда вы делаете **1** запрос за родителями, а потом в цикле дёргаете `p.children` — и ORM делает ещё **N** запросов (по одному на родителя). Итого `N+1` сетевых обращений вместо двух.

Как поймать: `engine.echo=True` — в логах видно один `SELECT ... FROM parents` и N однотипных `SELECT ... FROM children WHERE parent_id = ...`.

Как лечить:
- `selectinload(Parent.children)` — ровно **2** запроса: `SELECT parents` + `SELECT children WHERE parent_id IN (...)`. Часто рекомендуемый дефолт.
- `joinedload(Parent.children)` — **1** запрос с `LEFT OUTER JOIN`, но с дублями родительских колонок. На широких таблицах — лишний трафик. Требует `.unique()` при использовании с `scalars().all()`.


In [16]:
from sqlalchemy import event
from sqlalchemy.orm import selectinload

def count_queries(fn):
    """Считает SQL-запросы, которые выполнит fn()."""
    counter = {"n": 0}
    def before(conn, cursor, statement, params, ctx, many):
        counter["n"] += 1
    event.listen(engine, "before_cursor_execute", before)
    try:
        result = fn()
    finally:
        event.remove(engine, "before_cursor_execute", before)
    return result, counter["n"]

# Готовим данные: 5 пилотов, у каждого 2 самолёта
with Session(engine) as s:
    s.query(Pilot).delete()
    s.query(Aircraft).delete()
    s.commit()
    for i in range(5):
        a1 = Aircraft(aircraft_code=f"A{i}1", model=f"Model {i}-1")
        a2 = Aircraft(aircraft_code=f"A{i}2", model=f"Model {i}-2")
        s.add(Pilot(name=f"Pilot_{i}", aircrafts=[a1, a2]))
    s.commit()

def lazy_load():
    with Session(engine) as s:
        pilots = s.scalars(select(Pilot)).all()
        for p in pilots:
            _ = p.aircrafts
        return len(pilots)

def eager_load():
    with Session(engine) as s:
        stmt = select(Pilot).options(selectinload(Pilot.aircrafts))
        pilots = s.scalars(stmt).all()
        for p in pilots:
            _ = p.aircrafts
        return len(pilots)

_, n_lazy  = count_queries(lazy_load)
_, n_eager = count_queries(eager_load)
print(f"ленивая (N+1)     : {n_lazy} запросов")
print(f"selectinload      : {n_eager} запросов")


ленивая (N+1)     : 6 запросов
selectinload      : 2 запросов


### Упражнение 8. `joinedload` и `.unique()`

**Задание.** Решите ту же задачу через `joinedload` и убедитесь, что запрос действительно один.

**Шаги.**
1. Импортируйте `joinedload` из `sqlalchemy.orm`.
2. Замените `selectinload(Pilot.aircrafts)` на `joinedload(Pilot.aircrafts)`.
3. Запустите `eager_load()` — получите `InvalidRequestError` (нужна дедупликация).
4. Исправьте: `s.scalars(stmt).unique().all()`.
5. Пересчитайте запросы через `count_queries`.

**Критерий успеха.** 1 запрос вместо 2. В комментарии — почему `selectinload` всё равно рекомендуют чаще.


In [17]:
# УПРАЖНЕНИЕ 8. joinedload: тот же eager-эффект, но ОДНИМ запросом
from sqlalchemy.exc import InvalidRequestError
from sqlalchemy.orm import joinedload

# --- Шаги 1-3. Сначала «забываем» .unique() и ловим ошибку ------------
def joined_no_unique():
    with Session(engine) as s:
        stmt = select(Pilot).options(joinedload(Pilot.aircrafts))
        return s.scalars(stmt).all()   # ❌ без .unique()

try:
    joined_no_unique()
except InvalidRequestError as e:
    print("Без .unique() SQLAlchemy ругается:", str(e).splitlines()[0])

# --- Шаги 4-5. Правильная версия и подсчёт запросов --------------------
def eager_load_joined():
    with Session(engine) as s:
        # joinedload приклеивает коллекцию через LEFT OUTER JOIN:
        # SELECT ... FROM pilots LEFT JOIN pilot_aircraft ... LEFT JOIN aircrafts_orm ...
        stmt = select(Pilot).options(joinedload(Pilot.aircrafts))
        # JOIN размножает строки: 5 пилотов x 2 самолёта = 10 строк результата.
        # .unique() дедуплицирует объекты Pilot по PK — это явное «подтверждение»,
        # что мы понимаем: строк в результате больше, чем объектов.
        pilots = s.scalars(stmt).unique().all()
        for p in pilots:
            _ = p.aircrafts   # коллекции уже загружены — дополнительных запросов НЕТ
        return len(pilots)

res, n_joined = count_queries(eager_load_joined)
print(f"joinedload: {n_joined} запрос(ов), уникальных пилотов: {res}")   # 1 вместо 2

# ПОЧЕМУ selectinload ВСЁ ЖЕ РЕКОМЕНДУЮТ ЧАЩЕ (комментарий к заданию):
# 1) joinedload дублирует колонки родителя в КАЖДОЙ строке JOIN — на широких
#    таблицах это заметный лишний трафик;
# 2) если joinedload'ить сразу ДВЕ коллекции, строки размножаются декартово (M x N);
# 3) для коллекций joinedload требует .unique() — легко забыть и поймать ошибку
#    в рантайме (см. блок try/except выше);
# 4) два простых запроса selectinload (по PK + parent_id IN (...)) дают
#    предсказуемые планы и читаемые логи, а лишний round-trip обычно дешевле
#    раздутого ответа.
# joinedload берут, когда важен КАЖДЫЙ round-trip (медленная сеть), а коллекция небольшая.


Без .unique() SQLAlchemy ругается: The unique() method must be invoked on this Result, as it contains results that include joined eager loads against collections
joinedload: 1 запрос(ов), уникальных пилотов: 5


## Итоги семинара

1. **psycopg:** соединения через `with`, курсоры через `with`, всегда параметризация `%s`.
2. **Транзакции:** `with conn:` → commit/rollback; короткие транзакции; DDL транзакционен.
3. **SQL-инъекции:** f-string в SQL — уязвимо; `%s` и `:name` — безопасно.
4. **executemany:** один вызов вместо N `execute`; для миллионов строк — `COPY`.
5. **SQLAlchemy Core:** `text()` + `engine.connect()` + `executemany` из списка словарей.
6. **Pandas:** `read_sql(chunksize=…)` и `np.where` — векторизация; PostgreSQL для JOIN, pandas — для фич.
7. **ORM:** `DeclarativeBase`, `relationship(secondary=…)`, `ondelete='CASCADE'` на уровне БД.
8. **N+1:** ловится `echo=True`, лечится `selectinload` (обычно) или `joinedload().unique()` (когда важна экономия на количестве запросов).


---

# Решения

> Дальше — эталонные решения всех 8 упражнений. **Не подглядывайте, пока не попробовали.**


In [18]:
# РЕШЕНИЕ 1
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT id, name, qty FROM demo_items WHERE name = %s", ("apple",))
        print("apple :", cur.fetchall())
        cur.execute("SELECT id, name, qty FROM demo_items WHERE name = %s", ("apple' OR '1'='1",))
        print("inject:", cur.fetchall())  # []


apple : [(1, 'apple', 10)]
inject: []


In [19]:
# РЕШЕНИЕ 2
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS ex2")
        cur.execute("CREATE TABLE ex2 (id INT PRIMARY KEY, name TEXT)")
    conn.commit()

try:
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            cur.executemany("INSERT INTO ex2 VALUES (%s, %s)", [(1, "a"), (2, "b"), (1, "dup")])
        conn.commit()
except UniqueViolation as e:
    print("caught:", str(e).splitlines()[0])

with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT count(*) FROM ex2")
        print("rows after rollback:", cur.fetchone()[0])  # 0


caught: duplicate key value violates unique constraint "ex2_pkey"
rows after rollback: 0


In [20]:
# РЕШЕНИЕ 3
def fixed(code):
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT id, name, qty FROM demo_items WHERE name = %s LIMIT 5", (code,))
            return cur.fetchall()

print("broken:", broken("' OR '1'='1"))
print("fixed :", fixed("' OR '1'='1"))


broken: [(1, 'apple'), (2, 'banana')]
fixed : []


In [21]:
# РЕШЕНИЕ 4
import time, random

N = 10_000
rows = [(i, f"item_{i}", random.randint(1, 100)) for i in range(1, N + 1)]

with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS bench10k")
        cur.execute("CREATE TABLE bench10k (id INT PRIMARY KEY, name TEXT, val INT)")
    conn.commit()

t0 = time.perf_counter()
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        for r in rows:
            cur.execute("INSERT INTO bench10k VALUES (%s, %s, %s)", r)
    conn.commit()
t_loop = time.perf_counter() - t0

with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("TRUNCATE bench10k")
    conn.commit()

t0 = time.perf_counter()
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.executemany("INSERT INTO bench10k VALUES (%s, %s, %s)", rows)
    conn.commit()
t_batch = time.perf_counter() - t0

print(f"loop={t_loop:.2f}s  batch={t_batch:.2f}s  speedup=x{t_loop/max(t_batch,1e-9):.1f}")
# Почему не «в N раз быстрее» линейно: часть времени уходит на commit/WAL,
# а с ростом батча растёт время на один пакет; executemany уже почти линейно-эффективен.


loop=0.47s  batch=0.15s  speedup=x3.1


In [22]:
# РЕШЕНИЕ 5
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS ex5_measurements"))
    conn.execute(text("CREATE TABLE ex5_measurements (id INT PRIMARY KEY, sensor TEXT, value NUMERIC)"))
    conn.commit()
    records = [{"id": i, "sensor": f"S{i%4}", "value": i * 0.7} for i in range(1, 201)]
    conn.execute(
        text("INSERT INTO ex5_measurements (id, sensor, value) VALUES (:id, :sensor, :value)"),
        records,
    )
    conn.commit()
    cnt = conn.execute(text("SELECT count(*) FROM ex5_measurements WHERE value > :t"), {"t": 50}).scalar()
    print("count > 50:", cnt)


count > 50: 129


In [23]:
# РЕШЕНИЕ 6
import pandas as pd

agg = {}
for chunk in pd.read_sql("SELECT sensor, value FROM ex5_measurements", engine, chunksize=50):
    g = chunk.groupby("sensor").agg(mean=("value", "mean"), n=("value", "size"))
    for sensor, row in g.iterrows():
        if sensor not in agg:
            agg[sensor] = {"sum": 0.0, "n": 0}
        agg[sensor]["sum"] += float(row["mean"]) * int(row["n"])
        agg[sensor]["n"]   += int(row["n"])

for s, a in sorted(agg.items()):
    print(f"{s}: mean={a['sum']/a['n']:.2f}, n={a['n']}")
print("total n:", sum(a["n"] for a in agg.values()))


S0: mean=71.40, n=50
S1: mean=69.30, n=50
S2: mean=70.00, n=50
S3: mean=70.70, n=50
total n: 200


In [24]:
# РЕШЕНИЕ 7
from sqlalchemy import ForeignKey, String, Table, Column, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session

class Base2(DeclarativeBase):
    pass

author_book = Table(
    "author_book", Base2.metadata,
    Column("author_id", ForeignKey("authors.id", ondelete="CASCADE"), primary_key=True),
    Column("book_id",   ForeignKey("books.id",   ondelete="CASCADE"), primary_key=True),
)

class Author(Base2):
    __tablename__ = "authors"
    id:   Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    books: Mapped[list["Book"]] = relationship(secondary=author_book, back_populates="authors")

class Book(Base2):
    __tablename__ = "books"
    id:    Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(200))
    authors: Mapped[list["Author"]] = relationship(secondary=author_book, back_populates="books")

Base2.metadata.drop_all(engine)
Base2.metadata.create_all(engine)

with Session(engine) as s:
    tolstoy = Author(name="Толстой", books=[Book(title="Война и мир"), Book(title="Анна Каренина")])
    dosto   = Author(name="Достоевский", books=[Book(title="Идиот"), Book(title="Война и мир")])
    s.add_all([tolstoy, dosto])
    s.commit()
    print("links before:", len(s.execute(select(author_book)).fetchall()))

    s.delete(s.query(Author).filter_by(name="Толстой").first())
    s.commit()
    print("links after :", len(s.execute(select(author_book)).fetchall()))
    print("books left  :", s.query(Book).count())


links before: 4
links after : 2
books left  : 4


In [25]:
# РЕШЕНИЕ 8
from sqlalchemy.orm import joinedload

def eager_load_joined():
    with Session(engine) as s:
        stmt = select(Pilot).options(joinedload(Pilot.aircrafts))
        pilots = s.scalars(stmt).unique().all()
        for p in pilots:
            _ = p.aircrafts
        return len(pilots)

_, n_joined = count_queries(eager_load_joined)
print("joinedload queries:", n_joined)  # 1
# Комментарий: selectinload предпочтительнее для больших коллекций,
# потому что не дублирует родительские колонки в каждой строке JOIN.


joinedload queries: 1
